# Cosmic-ray transport equation solver

#### Imports

In [ ]:
using HDF5
using SpecialFunctions
using Gadfly
include("CRT_func_3D_4.jl")
include("losses_6.jl")

#### Constants

In [ ]:
const pc = 3e18;
const kpc = 3e21;
const yr = 365 * 24 * 3600;

#### Grid definition

In [ ]:
nr, nz, np = 40, 41, 80

r_min = 0.2*pc
r_max = 8*kpc

z_min = 0.2*pc
z_max = 4*kpc

p_min = 1.0
p_max = 1e9 # 1MeV to 1PeV

r_list, z_list, p_list, pu_list = make_grids(nr, nz, np, r_min, r_max, z_min, z_max, p_min, p_max);

#### Physical parameters

In [ ]:
m_p = 938.272 # in MeV, proton mass

D_0r = 6e29; # r-Diffusion coefficient, cm^2/s
D_0z = 6e29; # z-Diffusion coefficient, cm^2/s

E_0 = 1e7; # 10 TeV
p_0 = sqrt(E_0^2 + 2*m_p*E_0); # p cut-off (10 TeV), MeV
println("p_0 = ", p_0/1e6)
d = 0.3; # Diffusion index

v_w = 2e7; # Galactic wind velocity (200 km/s), cm/s v_min = 200 km/s and v_max = 1000 km/s (Crocker 2011)
v_A = 1e7; # Alfvèn speed, 100 km/s

r_cmz = 200*pc #Tsuboi, Ferrière, and the 1996 paper
z_cmz = 50*pc

V_cmz = pi * r_cmz^2 * (2 * z_cmz);
M_sol = 2e33; # Solar mass, g
m_H = 1.67e-24; # Hydrogen atom mass, g
m_avg = 1.4 * m_H; # Average hydrogen mass, g
M_cmz = 6e7 * M_sol; # CMZ mass, g

n_cmz = (M_cmz / m_avg) / V_cmz; # Particle density in the CMZ, cm^-3
println("Particle number density in the CMZ is ", n_cmz, " cm-3")
n_disk = 1.0; # Particle density in the disk, cm^-3
n_out = 1e-2; # Particle density outside the disk, cm^-3

#### Functions for diffusion, advection and losses

In [ ]:
function Dr(r::Float64, z::Float64, p::Float64)
    return D_0r * (p / p_0)^d 
end


function Dz(r::Float64, z::Float64, p::Float64)
    return D_0z * (p / p_0)^d
end


function Dp(r::Float64, z::Float64, p::Float64)
    return (p^2 * v_A^2) / (9 * Dr(r, z, p))
end


function v_z(z::Float64)
    if z > 0
        return v_w
    elseif z < 0
        return - v_w
    else
        return 0
    end
end


function P_dot_p(r::Float64, z::Float64, p::Float64)
    if r <= r_cmz && abs(z) <= z_cmz
        return p_p_dot(p, n_cmz)
    elseif r > r_cmz && abs(z) <= z_cmz
        return p_p_dot(p, n_disk)
    else 
        return p_p_dot(p, n_out)
    end
end

#### Source term

In [ ]:
Q_0 = 1.0
s = 2.4

function Q_gaussian(r::Float64, z::Float64, p::Float64, sigma_r::Float64, sigma_z::Float64)
    Q_term = Q_0 * (p / p_0)^(-s)
    norm_den = ((2*pi)^(3/2))*(sigma_r^2)*sigma_z
    exp_r_term = exp(-(r^2)/(2*sigma_r^2))
    exp_z_term = exp(-(z^2)/(2*sigma_z^2))
    return (Q_term / norm_den) * exp_r_term * exp_z_term
end

sigma_r = pc
sigma_z = pc

Q = zeros(length(r_list), length(z_list), length(p_list)) # nr, nz, np
Qu = zeros(length(r_list), length(z_list), length(pu_list)) # nr, nz, np-1

for k in eachindex(p_list)
    for j in eachindex(z_list)
        for i in eachindex(r_list)
            Q[i, j, k] = Q_gaussian(r_list[i], z_list[j], p_list[k], sigma_r, sigma_z) # Q(r_i, z_j, p_k)
        end
    end
end


for k in eachindex(pu_list)
    for j in eachindex(z_list)
        for i in eachindex(r_list)
            Qu[i, j, k] = Q_gaussian(r_list[i], z_list[j], pu_list[k], sigma_r, sigma_z) # Q(r_i, z_i, p_k+1)
        end
    end
end

#### Time step definition

In [ ]:
Dt_min = 1e-2*yr
m = 200
nmax = 1e5
Dt_list = make_time_grid(Dt_min, m, nmax);

### **Transport equation solver**

In [ ]:
prev_sol = zeros(nr, nz, np);
r_sol = zeros(nr, nz, np);
z_sol = zeros(nr, nz, np);
p_sol = zeros(nr, nz, np);

Dt = Dt_max 
t = 0

while Dt > Dt_min

    A_nextr, A_nr = make_Ar(r_list, z_list, p_list, Dr, Dt)
    A_nextz, A_nz = make_Az(r_list, z_list, p_list, Dz, v_z, Dt)
    A_nextp, A_np = make_Ap(r_list, z_list, p_list, P_dot_p, Dt)
    A_nextm, A_nm = make_Am(r_list, z_list, p_list, Dp, Dt)
    Q_arr = Q/4
    Qu_arr = Qu/4

    for n in 1:m
    
        global prev_sol, r_sol, z_sol, p_sol

        B_nr = make_Br(A_nr, prev_sol, Q_arr, Dt)
        Threads.@threads for k in 1:np
            @simd for j in 1:nz
                @inbounds r_sol[:, j, k] = A_nextr[j, k] \ B_nr[j, k, :]
            end
        end
        
        B_nz = make_Bz(A_nz, r_sol, Q_arr, Dt)
        Threads.@threads for k in 1:np
            @simd for i in 1:nr
                @inbounds z_sol[i, :, k] = A_nextz[i, k] \ B_nz[i, k, :]
            end
        end 

        B_np = make_Bp(A_np, z_sol, Q_arr, Qu_arr, Dt)
        Threads.@threads for j in 1:nz
            @simd for i in 1:nr
                @inbounds p_sol[i, j, :] = A_nextp[i, j] \ B_np[i, j, :]
            end
        end 

        B_nm = make_Bm(A_nm, p_sol, Q_arr, Dt)
        Threads.@threads for j in 1:nz
            @simd for i in 1:nr
                @inbounds prev_sol[i, j, :] = A_nextm[i, j] \ B_nm[i, j, :]
            end
        end 
    
    end

    t += m*Dt
    Dt =  Dt/1.1

end

println("tot_time = ", t/yr)

#### If plotting with matplotlib

You can save the data using HDF5:

In [ ]:
hdf5_file = h5open("psol_s2p4_6e7Msol.h5", "w")
write(hdf5_file, "psol_s2p4_6e7Msol", prev_sol)
close(hdf5_file)

And read it into a numpy array

In [ ]:
with h5py.File("psol_s2p4_6e7Msol.h5", 'r') as hdf_file:
    dataset = hdf_file["psol_s2p4_6e7Msol"]
    psol_numpy = np.array(dataset)

But pay attention to dimensions: (r, z, p) will be stored as (p, z, r)!